In [3]:
from datasets import load_dataset

In [4]:
mws_dataset = load_dataset("MERA-evaluation/LabTabVQA")
split = mws_dataset['test']

split[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/shots-00000-of-00001.parquet:   0%|          | 0.00/9.69M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/169M [00:00<?, ?B/s]

Generating shots split:   0%|          | 0/10 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/339 [00:00<?, ? examples/s]

{'instruction': 'Помогите мне, пожалуйста.\n\nЕсть задача такого типа. Задача на анализ изображений, содержащих табличные данные.\n\nИмеется 1 изображение\n\nИзображение: <image>\nВопрос:\n{question}\n\nA. {option_a}\nB. {option_b}\nC. {option_c}\nD. {option_d}\nE. {option_e}\nF. {option_f}\nG. {option_g}\n\nОпределите ответ к задаче, учитывая, что первому из предложенных вариантов ответа присваивается литера А, второму литера B, третьему литера C и так далее по английскому алфавиту. В качестве ответа выведите, пожалуйста, литеру, соответствующую верному варианту ответа из предложенных. Финальный ответ прошу написать после слова ОТВЕТ (литера через пробел после этого слова).',
 'inputs': {'image': {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x02\xfb\x00\x00\x01P\x08\x02\x00\x00\x00\xae\xd0&\xa9\x00\x01\x00\x00IDATx\x9ct\xfd\xf9\xb7%\xc9q\x1e\x08~f\xee\x11wyk\xae\x95Y+\xaaP\x05\x12$A\x8a\xa4\x96\x9e3-\xa9%\x9d\xd1L\xff\x05\xd3\x7f\xe8\x9c>sz;j\x89\x1a\x1e\x91\x12\xa9n\x90\x04

In [7]:
print(len(split))
print(split.column_names)

339
['instruction', 'inputs', 'outputs', 'meta']


In [8]:
import pprint
pprint.pprint({k: v for k, v in split[0].items() if k != "image"})

{'inputs': {'image': {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR'
                               b'\x00\x00\x02\xfb\x00\x00\x01P\x08\x02\x00\x00'
                               b'\x00\xae\xd0&\xa9\x00\x01\x00\x00IDATx\x9ct'
                               b'\xfd\xf9\xb7%\xc9q\x1e\x08~f\xee\x11wyk\xae'
                               b'\x95Y+\xaaP\x05\x12$A\x8a\xa4\x96\x9e3-\xa9'
                               b'%\x9d\xd1L\xff\x05\xd3\x7f\xe8\x9c>sz;j\x89'
                               b'\x1a\x1e\x91\x12\xa9n\x90\x04\x08\x10\x00Y'
                               b'\xa8%\xb7\x97\xf9\xd6\xbbD\x84\x9b\xcd\x0f'
                               b'f\xe6\xe1\xf7\x15\xfau\x0b\xccz\xef\xde'
                               b'\x08_\xcc\xcd>\xfblq\xfa\xd3?\xff3\x00\x0c('
                               b'\xa0\xf0\x1f\x812(\xfe\x0b\xaa\xaa\x00T'
                               b'\x81\xc2\x80*\x88\x88\x88\x01\x02@DP'
                               b'\x06\x89\x02d\xff\xd9\xfc\xa8\xfa\xb7\xe7\xdf'
       

In [14]:
from PIL import Image
import os, io, csv, shutil
from google.colab import files

BASE_DIR = "labtabvqa"
IMG_DIR  = f"{BASE_DIR}/images"
CSV_PATH = f"{BASE_DIR}/data.csv"
os.makedirs(IMG_DIR, exist_ok=True)

rows = []
for sample in split:
    inp  = sample["inputs"]
    meta = sample["meta"]
    cat  = meta["categories"]

    img_filename = f"{meta['id']:04d}.png"
    img_fullpath = f"{IMG_DIR}/{img_filename}"
    Image.open(io.BytesIO(inp["image"]["bytes"])).save(img_fullpath)

    rows.append({
        "id":              meta["id"],
        "img_path":        f"images/{img_filename}",
        "question":        inp["question"],
        "option_a":        inp["option_a"],
        "option_b":        inp["option_b"],
        "option_c":        inp["option_c"],
        "option_d":        inp["option_d"],
        "option_e":        inp["option_e"],
        "option_f":        inp["option_f"],
        "option_g":        inp["option_g"],
        "ground_truth":    sample["outputs"],
        "label":           "",
        "question_type":   cat["question_type"],
        "question_text":   cat["question_text"],
        "question_source": cat["question_source"],
        "rows":            meta["rows"],
        "columns":         meta["columns"],
    })

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Сохранено {len(rows)}")

shutil.make_archive("labtabvqa", "zip", ".", BASE_DIR)
files.download("labtabvqa.zip")

Сохранено 339


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>